# Credit Card Customer Segmentation
### Unsupervised Machine Learning: K-Means vs Hierarchical Clustering vs DBSCAN

---

## 1. Project Overview

**Business problem:** A bank has thousands of credit card customers, and they do not all behave the same way. Some spend a lot, some barely use the card, and some mostly withdraw cash. Sending the same offer to everyone is wasteful.

**Goal:** Group customers into segments based on their card usage behaviour, so the bank can target each group with the right offer.

**Why unsupervised learning?** We have no column that says "this customer is type A or type B". There is no target to predict. We let the algorithms find the groups on their own.

**What we will do in this project:**

| Step | What we do |
|------|-----------|
| 1 | Load and understand the data |
| 2 | Clean the data |
| 3 | Explore the data (EDA) |
| 4 | Prepare features (log transform and scaling) |
| 5 | Apply **K-Means** |
| 6 | Apply **Hierarchical Clustering** |
| 7 | Apply **DBSCAN** |
| 8 | Compare the three algorithms |
| 9 | Profile the segments and give business recommendations |

## 2. About the Dataset

**Source:** Credit Card Dataset for Clustering (Kaggle, by Arjun Bhasin)

The file contains about 9,000 active credit card holders observed over the last 6 months. Each row is one customer and each column describes their behaviour.

| Column | Meaning |
|--------|---------|
| CUST_ID | Customer ID |
| BALANCE | Amount left in the account |
| BALANCE_FREQUENCY | How often the balance is updated (0 to 1) |
| PURCHASES | Total amount spent on purchases |
| ONEOFF_PURCHASES | Largest single purchase amount |
| INSTALLMENTS_PURCHASES | Amount spent in instalments |
| CASH_ADVANCE | Cash withdrawn using the card |
| PURCHASES_FREQUENCY | How often purchases are made (0 to 1) |
| ONEOFF_PURCHASES_FREQUENCY | How often one-off purchases are made (0 to 1) |
| PURCHASES_INSTALLMENTS_FREQUENCY | How often instalment purchases are made (0 to 1) |
| CASH_ADVANCE_FREQUENCY | How often cash advance is taken (0 to 1) |
| CASH_ADVANCE_TRX | Number of cash advance transactions |
| PURCHASES_TRX | Number of purchase transactions |
| CREDIT_LIMIT | Credit limit of the customer |
| PAYMENTS | Total payments made by the customer |
| MINIMUM_PAYMENTS | Minimum payments due |
| PRC_FULL_PAYMENT | Percentage of full payment paid (0 to 1) |
| TENURE | Number of months as a card holder |

## 3. Import Libraries

We import the tools we need:
- **pandas, numpy** for data handling
- **matplotlib, seaborn** for charts
- **scikit-learn** for scaling, the three clustering algorithms, PCA and evaluation metrics
- **scipy** for drawing the dendrogram

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


## 4. Load and Understand the Data

First we load the CSV file and look at it. Always start by asking: how big is the data, what do the columns look like, and what are the data types?

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "CC GENERAL.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "arjunbhasin2013/ccdata",
  file_path,

)

df.head()

import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "CC GENERAL.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "arjunbhasin2013/ccdata",
  file_path,

)

df.head()

## 5. Data Cleaning

### 5.1 Check missing values

In [ ]:
df.isnull().sum()

### 5.2 Check duplicates and drop the ID column

`CUST_ID` is only a label. It carries no behaviour information, so we remove it before clustering. If we kept it, the algorithm would try to use it.

In [ ]:
df['CREDIT_LIMIT'].fillna(df['CREDIT_LIMIT'].median(), inplace = True)
df['MINIMUM_PAYMENTS'].fillna(df['MINIMUM_PAYMENTS'].median(), inplace = True)

df.isnull().sum()

df.duplicated().sum()
df.drop('CUST_ID', axis = 1)

df 

## 6. Exploratory Data Analysis (EDA)

### 6.1 Distribution of key features